# 02. LoRA Fine-Tuning

**Topics covered:** LoRA Theory · Rank · Alpha · PEFT Library

This notebook builds on [01_full_finetuning.ipynb](https://github.com/S33mi/modern-ai-llm-journey/blob/main/03_finetuning/01_full_finetuning.ipynb). Instead of updating every weight, we inject a small number of trainable low-rank matrices.

We will:
1. Understand the **LoRA** formulation (rank $r$, scaling $\alpha$)
2. Use the 🤗 **PEFT** library to wrap a model
3. Fine-tune only the LoRA adapters on a sentiment task
4. Compare parameter count and training behaviour with full fine-tuning
5. Save, reload and merge adapters

## 1. Setup & Imports

```bash
pip install transformers datasets evaluate accelerate peft scikit-learn
```

In [4]:
#!pip install transformers datasets evaluate accelerate peft scikit-learn

In [5]:
import torch
import numpy as np
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
)
from peft import (
    LoraConfig,
    get_peft_model,
    TaskType,
    PeftModel,
)
import evaluate

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

Using device: cpu


## 2. LoRA Theory

### The problem with full fine-tuning

A model with $d$ hidden dimensions has weight matrices $W \in \mathbb{R}^{d \times d}$.  
Updating every entry is expensive in memory (optimizer states) and storage (one full checkpoint per task).

### Low-Rank Adaptation (Hu et al., 2021)

Instead of learning $\Delta W$ directly, we parameterise the update as a product of two thin matrices:

$$
\Delta W = B A
$$

where

- $A \in \mathbb{R}^{r \times d}$
- $B \in \mathbb{R}^{d \times r}$
- $r \ll d$ (the **rank**)

The forward pass becomes:

$$
h = W_0 x + \frac{\alpha}{r} B A x
$$

- $W_0$ is frozen
- only $A$ and $B$ are trained
- $\alpha$ is a scaling hyper-parameter (often set equal to $r$ so the effective scale starts at 1)

### Why it works

Empirical evidence shows that the adaptation needed for many downstream tasks lies in a low-dimensional subspace. A rank of 8–64 is frequently enough.

### Rank ($r$) and Alpha ($\alpha$)

| Hyper-parameter | Typical values | Effect |
|-----------------|----------------|--------|
| **$r$ (rank)** | 4, 8, 16, 32, 64 | Higher → more capacity, more parameters |
| **$\alpha$ (lora_alpha)** | equal to $r$ or $2r$ | Scales the LoRA contribution; $\alpha/r$ is the effective multiplier |
| **lora_dropout** | 0.05–0.1 | Regularisation on the LoRA path |
| **target_modules** | `"q_proj"`, `"v_proj"`, … | Which linear layers receive adapters |

A common starting point for classification / instruction tuning is `r=8` or `r=16`, `lora_alpha=16` or `32`.

## 3. Dataset (same style as full fine-tuning)

You can keep using `acosio14/imbd-movie-reviews` or any IMDB-style dataset.  
Below we load a compact public subset so the notebook stays self-contained; swap the dataset name if you prefer your previous one.

In [6]:
# Option A – same dataset as full fine-tuning
raw = load_dataset("acosio14/imbd-movie-reviews")

# Option B – classic IMDB (small slice for quick runs)
# raw = load_dataset("imdb")
train_ds = raw["train"].shuffle(seed=42).select(range(2000))
eval_ds  = raw["test"].shuffle(seed=42).select(range(500))

print(train_ds)
print(train_ds[0])

README.md:   0%|          | 0.00/491 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 49.4MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 12.2MB            

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/40000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/10000 [00:00<?, ? examples/s]

Dataset({
    features: ['text', 'labels', 'input_ids', 'attention_mask'],
    num_rows: 2000
})
{'text': "Truly awful nonsensical garbage. This movie does everything wrong except make the running time under an hour. The gore FX defy gravity & logic. There are no scares. The acting is abysmal, with everyone appearing to be reading their lines. There's a surprise ending that's just silly where we find out that things we saw happen didn't even happen. Boy do I hate cop out endings! They pad this thing out with long drawn-out shots of people doing nothing interesting(like putting on make-up or talking for what seems like forever). They have to pad out a movie that's under an hour long? Ridiculous. The story itself is pretty freakin' thin. I mean it's just a variation of the movie APRIL FOOL'S DAY, if I remember that movie correctly, and that film wasn't all that great either. The only good thing I can say is it seems to have been shot well. Too bad nothing happens that's very exciting.", 

In [7]:
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, max_length=256)

train_tok = train_ds.map(tokenize, batched=True, remove_columns=["text"])
eval_tok  = eval_ds.map(tokenize, batched=True, remove_columns=["text"])

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

## 4. Create a LoRA Model with PEFT

In [9]:
# Upgrade torchao to a compatible version
!pip install -U "torchao>=0.16.0" --quiet

# (Optional but recommended) also make sure peft is recent
!pip install -U peft --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 41.1 MB/s eta 0:00:00


In [10]:
# Base model (all weights frozen later by PEFT)
base_model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2,
)

# LoRA configuration
lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,   # sequence classification
    r=8,                          # rank
    lora_alpha=16,                # scaling (alpha/r = 2)
    lora_dropout=0.1,
    target_modules=["q_lin", "v_lin"],  # DistilBERT attention projections
    bias="none",
)

model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


trainable params: 739,586 || all params: 67,694,596 || trainable%: 1.0925


You should see something like:

```
trainable params: ~0.3% of total
```

That is the whole point of LoRA – only a few hundred thousand parameters are updated.

In [11]:
# Inspect which modules received adapters
for name, param in model.named_parameters():
    if param.requires_grad:
        print(name, tuple(param.shape))

base_model.model.distilbert.transformer.layer.0.attention.q_lin.lora_A.default.weight (8, 768)
base_model.model.distilbert.transformer.layer.0.attention.q_lin.lora_B.default.weight (768, 8)
base_model.model.distilbert.transformer.layer.0.attention.v_lin.lora_A.default.weight (8, 768)
base_model.model.distilbert.transformer.layer.0.attention.v_lin.lora_B.default.weight (768, 8)
base_model.model.distilbert.transformer.layer.1.attention.q_lin.lora_A.default.weight (8, 768)
base_model.model.distilbert.transformer.layer.1.attention.q_lin.lora_B.default.weight (768, 8)
base_model.model.distilbert.transformer.layer.1.attention.v_lin.lora_A.default.weight (8, 768)
base_model.model.distilbert.transformer.layer.1.attention.v_lin.lora_B.default.weight (768, 8)
base_model.model.distilbert.transformer.layer.2.attention.q_lin.lora_A.default.weight (8, 768)
base_model.model.distilbert.transformer.layer.2.attention.q_lin.lora_B.default.weight (768, 8)
base_model.model.distilbert.transformer.layer.2.at

## 5. Metrics & TrainingArguments

In [12]:
accuracy = evaluate.load("accuracy")
f1 = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy.compute(predictions=preds, references=labels)["accuracy"],
        "f1": f1.compute(predictions=preds, references=labels, average="binary")["f1"],
    }

In [14]:
training_args = TrainingArguments(
    output_dir="./results-imdb-lora",
    num_train_epochs=2,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    learning_rate=3e-4,              # LoRA often tolerates a higher LR
    weight_decay=0.01,
    eval_strategy="epoch",          # ← changed from evaluation_strategy
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    logging_steps=50,
    fp16=torch.cuda.is_available(),
    report_to="none",
)

## 6. Train with Trainer

In [16]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=eval_tok,
    processing_class=tokenizer,   # ← changed from tokenizer=...
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

train_result = trainer.train()
print(train_result)

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
[transformers] `use_return_dict` is deprecated! Use `return_dict` instead!


Epoch,Training Loss,Validation Loss


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.326660,0.300480,0.878000,0.875764
2,0.278479,0.296328,0.884000,0.879167


/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


TrainOutput(global_step=500, training_loss=0.35425451278686526, metrics={'train_runtime': 4837.287, 'train_samples_per_second': 0.827, 'train_steps_per_second': 0.103, 'total_flos': 269011436378496.0, 'train_loss': 0.35425451278686526, 'epoch': 2.0})


In [17]:
metrics = trainer.evaluate()
print("Eval metrics:", metrics)

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Training Loss,Validation Loss,Epoch,Accuracy,F1
0.278479,0.296328,2,0.884000,0.879167


Eval metrics: {'eval_loss': 0.29632771015167236, 'eval_accuracy': 0.884, 'eval_f1': 0.8791666666666667}


## 7. Inference

In [ ]:
def predict(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=256).to(device)
    model.to(device)
    with torch.no_grad():
        logits = model(**inputs).logits
    pred = logits.argmax(-1).item()
    label = "positive" if pred == 1 else "negative"
    prob = torch.softmax(logits, dim=-1)[0, pred].item()
    return label, prob


examples = [
    "This movie was an absolute masterpiece. I loved every minute of it!",
    "Boring, predictable and a complete waste of time.",
    "The acting was decent but the plot made no sense.",
]

for text in examples:
    label, prob = predict(text)
    print(f"[{label} {prob:.2f}]  {text}")

## 8. Saving & Loading LoRA Adapters

PEFT saves **only the adapter weights** (a few MB) instead of the full model.

In [19]:
adapter_dir = "./my-imdb-lora-adapter"
model.save_pretrained(adapter_dir)
tokenizer.save_pretrained(adapter_dir)
print(f"Adapter saved to {adapter_dir}")

Adapter saved to ./my-imdb-lora-adapter


In [20]:
# Reload later
base = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)
model_loaded = PeftModel.from_pretrained(base, adapter_dir)
model_loaded.eval()
print("Adapter reloaded successfully")

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Adapter reloaded successfully


### Optional: merge adapters into the base model

For deployment you can bake the LoRA weights into the original matrices and obtain a regular model with no PEFT dependency.

In [21]:
# merged = model_loaded.merge_and_unload()   # returns a plain transformers model
# merged.save_pretrained("./my-imdb-lora-merged")
print("Uncomment the lines above when you want a fully merged checkpoint.")

Uncomment the lines above when you want a fully merged checkpoint.


## 9. Full Fine-Tuning vs LoRA – Quick Comparison

| Aspect | Full Fine-Tuning | LoRA |
|--------|------------------|------|
| Trainable parameters | 100% | ~0.1–1% |
| GPU memory | High (optimizer states for all weights) | Much lower |
| Checkpoint size | Full model (hundreds of MB – GB) | Adapter only (MBs) |
| Risk of forgetting | Higher | Lower (base stays frozen) |
| Typical LR | 1e-5 – 5e-5 | 1e-4 – 3e-4 |
| Quality | Often marginally higher | Very close for most tasks |

For models larger than ~1 B parameters, LoRA (or QLoRA) is the default choice.

## 10. Choosing Target Modules

Different architectures name their linear layers differently:

| Architecture | Common `target_modules` |
|--------------|-------------------------|
| BERT / DistilBERT | `"q_lin"`, `"v_lin"` (or `"query"`, `"value"`) |
| GPT-2 | `"c_attn"` |
| LLaMA / Mistral | `"q_proj"`, `"k_proj"`, `"v_proj"`, `"o_proj"` |
| T5 | `"q"`, `"v"` |

You can also pass `target_modules="all-linear"` (PEFT ≥ 0.7) to adapt every linear layer.

## 11. Summary

| Concept | Meaning |
|---------|--------|
| **LoRA** | Low-rank update $\Delta W = BA$ injected next to a frozen weight |
| **Rank $r$** | Inner dimension of $A$ and $B$; controls capacity |
| **Alpha $\alpha$** | Scaling factor; effective multiplier is $\alpha/r$ |
| **PEFT** | Hugging Face library that implements LoRA, Prefix-Tuning, IA³, … |
| **Adapter checkpoint** | Only the small trainable matrices – easy to share and swap |

### Canonical LoRA snippet

```python
from peft import LoraConfig, get_peft_model, TaskType

config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    target_modules=["q_lin", "v_lin"],
)
model = get_peft_model(base_model, config)
model.print_trainable_parameters()
```

---

**Next notebook:** [`03_qlora_instruction_tuning.ipynb`](https://github.com/S33mi/modern-ai-llm-journey/blob/main/03_finetuning/03_qlora_instruction_tuning.ipynb)
4-bit Quantization · QLoRA · Instruction Tuning

---

**For contribution and insihght:** [**S33mi**](https://github.com/S33mi)

Open to Data Analytics and ML/AL related opportunities